# Autograd from Scratch

This notebook walks through MiniCNN's pure-NumPy reverse-mode automatic differentiation engine.
No PyTorch. No CUDA. Just Python and NumPy.

By the end you will understand:
- How a computation graph is built during the forward pass
- How `backward()` traverses that graph in reverse topological order
- How to use the `Function` API to write your own differentiable ops
- How to wire layers together and train a small model

In [ ]:
import sys
sys.path.insert(0, '../src')  # locate minicnn package from notebooks/

import numpy as np
import matplotlib.pyplot as plt

## 1. The Tensor: data + grad + a backward hook

Every value in the computation graph is a `Tensor`.
Three things matter:

| Field | Role |
|---|---|
| `data` | the NumPy array holding the value |
| `grad` | accumulated gradient (filled in by `backward()`) |
| `_backward` | closure that knows how to back-prop through *this* op |
| `_prev` | the set of tensors this one depends on |


In [ ]:
from minicnn.nn.tensor import Tensor

a = Tensor([2.0], requires_grad=True)
b = Tensor([3.0], requires_grad=True)

c = a * b  # creates a new Tensor and wires a _backward closure

print('c.data  =', c.data)   # 6.0
print('c._op   =', c._op)    # 'mul'
print('c._prev =', {t.data for t in c._prev})  # {[2.], [3.]}

## 2. Forward pass builds the graph automatically

Every arithmetic operation records the parent tensors and a closure.  
No explicit graph construction needed — it happens as you compute.

In [ ]:
x = Tensor([1.0, 2.0, 3.0], requires_grad=True)
w = Tensor([0.5, -1.0, 2.0], requires_grad=True)

dot = (x * w).sum()   # dot product
print('dot =', dot.data)   # 1*0.5 + 2*(-1) + 3*2 = 4.5

## 3. Backward: topological sort then chain rule

`backward()` does two things:
1. Builds a topological ordering of all nodes reachable from the output.
2. Walks in reverse, calling each node's `_backward()` closure.

In [ ]:
dot.backward()   # seed gradient = 1.0 (scalar output)

print('d(dot)/dx =', x.grad)  # should be w = [0.5, -1.0, 2.0]
print('d(dot)/dw =', w.grad)  # should be x = [1.0,  2.0, 3.0]

## 4. How a _backward closure works

Let's look at multiplication manually to demystify the magic:

In [ ]:
# This is essentially what Tensor.__mul__ does:

a = Tensor([4.0], requires_grad=True)
b = Tensor([5.0], requires_grad=True)

out = Tensor(a.data * b.data, requires_grad=True)
out._prev = {a, b}
out._op = 'mul'

def _backward():
    # chain rule: d(a*b)/da = b, d(a*b)/db = a
    a._add_grad(out.grad * b.data)
    b._add_grad(out.grad * a.data)

out._backward = _backward

out.grad = np.array([1.0])   # seed
_backward()

print('a.grad =', a.grad)  # 5.0 (= b.data)
print('b.grad =', b.grad)  # 4.0 (= a.data)

## 5. Parameters and zero_grad

`Parameter` is just a `Tensor` with `requires_grad=True` by default.  
Before each training step, call `zero_grad()` to reset accumulated gradients.

In [ ]:
from minicnn.nn.tensor import Parameter

w = Parameter([[1.0, 2.0], [3.0, 4.0]], name='w')
print('requires_grad:', w.requires_grad)  # True
print('name:', w.name)

# Two backward passes accumulate:
x = Tensor([[1.0, 0.0]])
(x @ w).sum().backward()
(x @ w).sum().backward()  # grad accumulates!
print('accumulated grad:', w.grad)   # doubled

w.zero_grad()
print('after zero_grad:', w.grad)    # None

## 6. Available ops

MiniCNN's `Tensor` supports:

In [ ]:
x = Tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)

print('relu:   ', x.relu().data)
print('sigmoid:', x.sigmoid().data)
print('tanh:   ', x.tanh().data)

# All have correct gradients:
x.zero_grad()
x.sigmoid().sum().backward()
# sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))
s = 1 / (1 + np.exp(-x.data))
expected = s * (1 - s)
print('sigmoid grad error:', np.abs(x.grad - expected).max())  # ~0

## 7. Building a model with layers

`Module` and `Sequential` follow the same pattern as PyTorch.

In [ ]:
from minicnn.nn import Linear, ReLU, Sigmoid, Sequential

model = Sequential(
    Linear(2, 8),
    ReLU(),
    Linear(8, 1),
    Sigmoid(),
)

x = Tensor(np.random.randn(4, 2).astype('float32'))
out = model(x)
print('output shape:', out.shape)   # (4, 1)
print('all in [0,1]:', (out.data >= 0).all() and (out.data <= 1).all())

## 8. Training loop: XOR problem

XOR is the simplest non-linearly-separable problem. Let's train it from scratch.

In [ ]:
from minicnn.nn.tensor import cross_entropy
from minicnn.optim.sgd import SGD
from minicnn.nn import Linear, ReLU, Sequential

# XOR dataset
X = Tensor(np.array([[0,0],[0,1],[1,0],[1,1]], dtype='float32'))
y = np.array([0, 1, 1, 0])  # labels

model = Sequential(Linear(2, 8), ReLU(), Linear(8, 2))
opt = SGD(model.parameters(), lr=0.1)

losses = []
for epoch in range(200):
    model.zero_grad()
    logits = model(X)
    loss = cross_entropy(logits, y)
    loss.backward()
    opt.step()
    losses.append(float(loss.data))

plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('XOR training')
plt.show()
print('final loss:', losses[-1])

In [ ]:
# Check predictions
from minicnn.nn.tensor import no_grad

with no_grad():
    logits = model(X)
    preds = logits.data.argmax(axis=1)

print('Predictions:', preds)
print('Targets:    ', y)
print('Accuracy:   ', (preds == y).mean())

## 9. Dropout and training/eval mode

`Dropout` is active in `training=True` mode and passes through unchanged in `eval()` mode.

In [ ]:
from minicnn.nn import Dropout

drop = Dropout(p=0.5)
x = Tensor(np.ones((1, 10), dtype='float32'))

drop.train(True)
out_train = drop(x)
print('train mode (zeros expected ~50%):', out_train.data)

drop.eval()
out_eval = drop(x)
print('eval mode (all ones):            ', out_eval.data)

## 10. Custom differentiable ops: the Function API

Subclass `Function` and implement `forward` + `backward`.
Call via `MyOp.apply(*inputs)` — the graph wiring is automatic.

In [ ]:
from minicnn.autograd.function import Function

class Swish(Function):
    """x * sigmoid(x) — smooth alternative to ReLU."""

    @staticmethod
    def forward(ctx, x):
        s = 1.0 / (1.0 + np.exp(-x.data))
        ctx.save_for_backward(x)
        ctx.s = s
        return Tensor(x.data * s, requires_grad=x.requires_grad)

    @staticmethod
    def backward(ctx, grad):
        (x,) = ctx.saved_tensors
        s = ctx.s
        return grad * (s + x.data * s * (1.0 - s))

x = Tensor([0.0, 1.0, 2.0], requires_grad=True)
y = Swish.apply(x)
print('Swish forward:', y.data)

y.sum().backward()
print('Swish gradient:', x.grad)

# Numerical gradient check
eps = 1e-4
numerical = []
for i in range(len(x.data)):
    xp = x.data.copy(); xp[i] += eps
    xm = x.data.copy(); xm[i] -= eps
    swish = lambda v: v * (1 / (1 + np.exp(-v)))
    numerical.append((swish(xp[i]) - swish(xm[i])) / (2 * eps))
print('Numerical grad:', np.array(numerical))
print('Max error:     ', np.abs(x.grad - numerical).max())

## 11. Connecting to the compiler + runtime pipeline

The `InferencePipeline` takes a YAML-style model config dict, traces it through the compiler (fusion detection, identity removal), and runs forward inference with the autograd layers.

This is the bridge between the static compiler and the dynamic runtime.

In [ ]:
from minicnn.runtime.pipeline import InferencePipeline

model_cfg = {
    'layers': [
        {'type': 'Linear', 'in_features': 4, 'out_features': 16},
        {'type': 'ReLU'},
        {'type': 'Linear', 'in_features': 16, 'out_features': 3},
    ]
}

pipeline = InferencePipeline.from_config(model_cfg, profile=True)

x = np.random.randn(8, 4).astype('float32')
logits = pipeline.run_final(x)
print('logits shape:', logits.shape)   # (8, 3)
print('profile:', pipeline.profile_summary())

In [ ]:
# Inspect the IR graph before and after optimization
from minicnn.compiler import trace_model_config, optimize
import json

conv_cfg = {
    'layers': [
        {'type': 'Conv2d', 'in_channels': 1, 'out_channels': 8, 'kernel_size': 3, 'padding': 1},
        {'type': 'BatchNorm2d', 'num_features': 8},
        {'type': 'ReLU'},
        {'type': 'Identity'},
        {'type': 'Flatten'},
        {'type': 'Linear', 'in_features': 8*28*28, 'out_features': 10},
    ]
}

raw_ir = trace_model_config(conv_cfg)
opt_ir = optimize(raw_ir)

print('Raw nodes:', [n.op for n in raw_ir.nodes])
print('Optimized:', [n.op for n in opt_ir.nodes])  # Identity removed

# Check fusion annotation
fused = [n for n in opt_ir.nodes if 'fusion_pattern' in n.attrs]
print('Fusion candidates:', [(n.name, n.attrs['fusion_pattern']) for n in fused])

## 12. Adam optimizer and gradient clipping

In [ ]:
from minicnn.optim.adam import Adam
from minicnn.nn import Linear, ReLU, Sequential
from minicnn.nn.tensor import cross_entropy

# Spiral dataset (2-class)
np.random.seed(42)
N = 100
theta = np.linspace(0, 4*np.pi, N)
r = np.linspace(0.2, 1.0, N)
X0 = np.stack([r*np.cos(theta), r*np.sin(theta)], axis=1)
X1 = np.stack([r*np.cos(theta+np.pi), r*np.sin(theta+np.pi)], axis=1)
X_data = np.vstack([X0, X1]).astype('float32')
y_data = np.array([0]*N + [1]*N)

model = Sequential(Linear(2, 32), ReLU(), Linear(32, 32), ReLU(), Linear(32, 2))
opt = Adam(model.parameters(), lr=1e-2, grad_clip=1.0)

losses = []
for epoch in range(300):
    model.zero_grad()
    logits = model(Tensor(X_data))
    loss = cross_entropy(logits, y_data)
    loss.backward()
    opt.step()
    losses.append(float(loss.data))

plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('cross-entropy loss'); plt.title('Spiral — Adam + grad_clip')
plt.show()
print('final loss:', round(losses[-1], 4))

## Next steps

- `minicnn train-autograd --config configs/autograd_tiny.yaml` — train from the CLI
- `minicnn train-flex --config templates/mnist/lenet_like.yaml` — full MNIST with PyTorch
- `docs/08_autograd.md` — full API reference (Function, optimizers, scheduler, datasets)
- `docs/architecture.md` — system diagram and module map
- `src/minicnn/autograd/function.py` — read the Function.apply() implementation